In [1]:
import tweepy
import pandas as pd
from datetime import datetime, timedelta

In [3]:
# Twitter API setup
BEARER_TOKEN = 'AAAAAAAAAAAAAAAAAAAAAIu%2F2wEAAAAAEVP6uITbFvEM8Vr4%2BZhAyvCMugE%3DmijxsvYvqORwpCDSdIAlcTNSAhan7A86VwcxSJrerILvlNRvmf'  
client = tweepy.Client(bearer_token=BEARER_TOKEN, wait_on_rate_limit=True)

In [19]:
query = '("Real Madrid Dortmund" OR #RealMadrid OR #Dortmund OR #ClubWorldCup) lang:en -is:retweet'

In [21]:
#Match time window: July 7, 2025 – 20:00 UTC kickoff
start_time = datetime(2025, 7, 7, 19, 0)  # One hour before kickoff
end_time = datetime(2025, 7, 7, 23, 0)    # Match ends + short buffer

In [23]:
start_str = start_time.isoformat("T") + "Z"
end_str = end_time.isoformat("T") + "Z"

In [ ]:
# Fetch tweets
tweets = []
for resp in tweepy.Paginator(client.search_recent_tweets,
                             query=query,
                             start_time=start_str,
                             end_time=end_str,
                             tweet_fields=['created_at', 'text', 'public_metrics'],
                             max_results=100):
    if resp.data:
        print(f"Fetched {len(resp.data)} tweets")
        tweets.extend(resp.data)
    else:
        print("No tweets in this response")

# Convert to DataFrame
df = pd.DataFrame([{
    'timestamp': t.created_at,
    'text': t.text,
    'likes': t.public_metrics['like_count'],
    'retweets': t.public_metrics['retweet_count']
} for t in tweets])

# Save to CSV
df.to_csv(r"H:\FOM- Study Material\Semester Three\Big-Data-Analysis Project\fam_momentum_index\Data\raw\realmadrid_dortmund_2025_qf.csv", index=False)


In [13]:
pip install pydub numpy pandas


In [1]:
import os
from pydub import AudioSegment
import numpy as np
import pandas as pd

data_path = r"H:\FOM- Study Material\Semester Three\Big-Data-Analysis Project\fam_momentum_index\Data"
audio_file = os.path.join(data_path, "RMA__vs__BVB_Full_TNT.mp3")  # Rename your MP3 file accordingly
output_file = os.path.join(data_path, "match_audio_volume_by_second.csv")

# Load audio
audio = AudioSegment.from_mp3(audio_file)

# Process every second
volume_data = []
for i in range(0, len(audio), 1000):
    segment = audio[i:i+1000]
    rms = segment.rms
    db = 20 * np.log10(rms) if rms > 0 else -100
    timestamp = i // 1000  # seconds
    volume_data.append({"second": timestamp, "rms_db": db})

# Create DataFrame
df = pd.DataFrame(volume_data)
df["normalized_volume"] = 1 - (df["rms_db"] - df["rms_db"].min()) / (df["rms_db"].max() - df["rms_db"].min())

# Save to CSV
df.to_csv(output_file, index=False)
print(f"Audio volume data saved to:\n{output_file}")


Audio volume data saved to:
H:\FOM- Study Material\Semester Three\Big-Data-Analysis Project\fam_momentum_index\Data\match_audio_volume_by_second.csv


In [1]:
import pandas as pd
import matplotlib.pyplot as plt


In [19]:
df = pd.read_csv(r"H:\FOM- Study Material\Semester Three\Big-Data-Analysis Project\fam_momentum_index\Data\tweet_sentiment_by_minute.csv")

In [21]:
print(df.columns)

Index(['minute;tweet_volume;avg_sentiment;norm_volume;norm_sentiment'], dtype='object')
